# Orders - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_orders"
target_table = f"{catalog}.silver.olist_orders"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.cs

In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 99441
Number of columns: 15


In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


In [0]:
for column, dtype in bronze_df.dtypes[:8]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

order_id
Null count: 0
Distinct count: 99441
Extra whitespace row count: 0
--------------------
customer_id
Null count: 0
Distinct count: 99441
Extra whitespace row count: 0
--------------------
order_status
Null count: 0
Distinct count: 8
Extra whitespace row count: 0
--------------------
order_purchase_timestamp
Null count: 0
Distinct count: 98875
--------------------
order_approved_at
Null count: 160
Distinct count: 90733
--------------------
order_delivered_carrier_date
Null count: 1783
Distinct count: 81018
--------------------
order_delivered_customer_date
Null count: 2965
Distinct count: 95664
--------------------
order_estimated_delivery_date
Null count: 0
Distinct count: 459
--------------------


- order_id is complete and unique. It is the key for this table.
- customer_id (foreign key) is also complete and unique in this table.
- No string trimming is needed.

In [0]:
display(bronze_df.groupBy("order_status").count())

order_status,count
delivered,96478
invoiced,314
shipped,1107
processing,301
unavailable,609
canceled,625
created,5
approved,2


In [0]:
display(bronze_df.filter(col("order_approved_at").isNull()).groupBy("order_status").count())

order_status,count
canceled,141
delivered,14
created,5


In [0]:
display(bronze_df.filter(col("order_delivered_carrier_date").isNull()).groupBy("order_status").count())

order_status,count
invoiced,314
processing,301
unavailable,609
canceled,550
created,5
approved,2
delivered,2


In [0]:
display(bronze_df.filter(col("order_delivered_customer_date").isNull()).groupBy("order_status").count())

order_status,count
invoiced,314
shipped,1107
processing,301
unavailable,609
canceled,619
delivered,8
created,5
approved,2


Most missing timestamps belong to orders that were never approved, shipped, or delivered. A few delivered orders also have missing timestamps, but there is not enough information to treat them as invalid. Therefore, all rows and timestamp nulls are preserved in Silver.

## Transform to Silver

No transformation is required.

## Write to Silver

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.csv,2026-08-02T21:30:14.000Z,2026-08-02T22:56:53.002Z,b3bb03d6-172c-4ce6-b6f5-bb440d200284,olist,orders
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,null,/Volumes/ecommerce_dev/landing/raw_files/olist/orders/olist_orders_dataset.cs

In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 99441
Silver row count: 99441
